In [56]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)


Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [57]:
import pandas as pd
from datetime import datetime

# Define time period
start = "2025-04-01T00:00:00Z"  # adjust as needed

# List of owners to pull from
list_of_owners = ['HTOC Org', 'HTOC Comm', 'HTOC legacy data']
final_results = []
summary = '%alynforensics%' #indicator
type_names = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
    "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"
]

# Convert the list to a comma-separated string for the TQL query
type_name_condition = ", ".join([f'"{type_name}"' for type_name in type_names])

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        # Build TQL query string
        tql = (
            f'ownerName EQ "{owner}" and '
            f'typeName IN ({type_name_condition}) and '
            f'(indicatorActive EQ true OR indicatorActive EQ false) and '
            f'summary LIKE "{summary}"'
        )

        # Set up the request
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators?tql={tql}&fields=threatAssess,associatedGroups,tags,observations,&resultStart=0&resultLimit=10000')

        # Send the request
        response = tc.api_request(ro)

        # Parse response
        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")

    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean results
if final_results:
    # Extract and normalize data only if 'data' key exists and contains 'summary'
    normalized_data = []
    for result in final_results:
        if 'data' in result:
            for item in result['data']:
                if 'summary' in item:
                    normalized_data.append(item)

    if normalized_data:
        observed_src = pd.json_normalize(normalized_data)
        observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
        observed_src = observed_src.drop_duplicates(subset='indicator', keep='first')  # Remove duplicates based on 'indicator'
        print(f"\nRetrieved {len(observed_src)} unique observed indicators.")
    else:
        print("\nNo valid indicators with 'summary' key retrieved.")
else:
    print("\nNo indicators retrieved.")



Querying owner: HTOC Org

Querying owner: HTOC Comm

Querying owner: HTOC legacy data

Retrieved 4 unique observed indicators.


In [58]:
observed_src

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,dnsActive,whoisActive,legacyLink,tags.data,associatedGroups.data,source,url,text,address,indicator
0,6755399443022985,2025-03-28T13:51:53Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-04-24T13:46:43Z,5.0,98,2.5,...,False,False,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 33375, 'name': 'benign', 'lastUsed': '...","[{'id': 6755399443003054, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,alynforensics.com
1,6755399443022988,2025-03-28T13:52:06Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-04-24T11:26:22Z,5.0,100,2.5,...,NaN,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 33375, 'name': 'benign', 'lastUsed': '...","[{'id': 6755399443003054, 'dateAdded': '2025-0...",https://alynforensics.com/OBS_Testing/Test.html,alynforensics.com/obs_testing/test.html,NaN,NaN,alynforensics.com/obs_testing/test.html
2,6755399443022987,2025-03-28T13:51:53Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,URL,2025-04-24T11:25:43Z,5.0,100,2.5,...,NaN,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 33375, 'name': 'benign', 'lastUsed': '...","[{'id': 6755399443003054, 'dateAdded': '2025-0...",NaN,NaN,https://alynforensics.com/OBS_Testing/Test.html,NaN,https://alynforensics.com/OBS_Testing/Test.html
3,6755399443022986,2025-03-28T13:51:53Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,EmailAddress,2025-04-24T11:25:20Z,5.0,100,2.5,...,NaN,NaN,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 33375, 'name': 'benign', 'lastUsed': '...","[{'id': 6755399443003054, 'dateAdded': '2025-0...",NaN,NaN,NaN,testing@alynforensics.com,testing@alynforensics.com


In [59]:
# Initialize an empty DataFrame to store filtered tags
filtered_tags = pd.DataFrame()

# Loop through each row in observed_src
for _, row in observed_src.iterrows():
    if 'tags.data' in row and isinstance(row['tags.data'], list):
        # Normalize the tags data for the current row
        tags = pd.json_normalize(row['tags.data'])
        
        # Add 'summary', 'observations', and 'description' columns to the tags
        tags['summary'] = row['summary']
        tags['observations'] = row.get('observations', None)
        tags['description'] = row.get('description', None)
        
        # Filter tags containing 'API' (case-insensitive)
        filtered = tags[tags['name'].str.contains('API', case=False, na=False)]
        
        # Append the filtered tags to the result DataFrame
        filtered_tags = pd.concat([filtered_tags, filtered], ignore_index=True)

# Display the filtered tags
filtered_tags


,id,name,lastUsed,description,summary,observations
0,23630,NIH Splunk API,2025-04-24T17:48:46Z,This indicator is benign and being used to tes...,alynforensics.com,2


In [60]:
# Define the file path
file_path = r'C:\Users\jaskew\Documents\project_repository\data\tracking\alynforensics.csv'

# Check if the file exists
if os.path.exists(file_path):
    # Check if the file is empty
    if os.path.getsize(file_path) > 0:
        # Load existing data
        existing_data = pd.read_csv(file_path)
        
        # Concatenate new data with existing data
        combined_data = pd.concat([existing_data, filtered_tags], ignore_index=True)
        
        # Drop duplicate records
        combined_data = combined_data.drop_duplicates()
    else:
        # If the file is empty, use the new data directly
        combined_data = filtered_tags
else:
    # If the file doesn't exist, use the new data directly
    combined_data = filtered_tags

# Save the combined data to the file
combined_data.to_csv(file_path, index=False)